# WC_BADGE_PRODUCT_D ETL Process
### Product Dimension - ODI to Databricks Migration
**Source Table:** `workspace.PRXBI_TS.WC_MERCURY_PRODUCT_TS`
**Target Table:** `workspace.PRXBI_DW.wc_badge_product_d`
**Detection Strategy:** NOT_EXISTS (full column CDC)
**DATASOURCE_NUM_ID:** 380

#### Migration Notes
- Oracle `PRXBI_DW_SEP` mapped to `workspace.PRXBI_DW`
- Oracle `PRXBI_TS_SEP` mapped to `workspace.PRXBI_TS`
- `SYSTIMESTAMP` replaced with `CURRENT_TIMESTAMP()`
- `NVL()` replaced with `COALESCE()`
- Oracle `/*+ append */` hints and `NOLOGGING` removed
- Oracle indexes removed (Delta handles via Z-ORDER)
- `DBMS_STATS` replaced with `OPTIMIZE` + `ZORDER`
- Oracle sequences (`SEQ.NEXTVAL`) replaced with `BIGINT GENERATED ALWAYS AS IDENTITY`
- Separate UPDATE + INSERT replaced with MERGE INTO
- NULL-safe comparison uses Spark `<=>` operator
- All tables use Delta format

In [ ]:
%sql
-- Step 1: Create Widgets for ETL Parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'FULL';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '0';

## Step 2: ETL Parameter Extraction
Read `wc_etl_parameters` to obtain extract time windows and current ROW_WID.

In [ ]:
%sql
-- Step 2a: Create temp views for ETL parameters
CREATE OR REPLACE TEMP VIEW v_etl_params AS
SELECT
  COALESCE(MAX(CASE WHEN PARAM_NAME = 'LAST_EXTRACT_TIME' THEN PARAM_VALUE END),
           CAST('1900-01-01 00:00:00' AS TIMESTAMP)) AS last_extract_time,
  CURRENT_TIMESTAMP() AS current_extract_time
FROM workspace.PRXBI_DW.wc_etl_parameters
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID};

CREATE OR REPLACE TEMP VIEW v_row_wid AS
SELECT COALESCE(MAX(ROW_WID), 0) AS max_row_wid
FROM workspace.PRXBI_DW.wc_badge_product_d;

In [ ]:
%sql
-- Step 2b: Display ETL parameters for validation
SELECT * FROM v_etl_params;
SELECT * FROM v_row_wid;

## Step 3: Create C$ Staging Table
Drop and recreate the staging table `c_0filter_stg` to hold deduplicated source records.
Uses `ROW_NUMBER() OVER(PARTITION BY ID ORDER BY INT_INSERT_DATE DESC)` to take the latest record per product ID.

In [ ]:
%sql
-- Step 3a: Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0filter_stg;

In [ ]:
%sql
-- Step 3b: Create staging table
CREATE TABLE workspace.PRXBI_DW.c_0filter_stg (
  NAME               STRING,
  PRICE              DECIMAL(18,4),
  STATUS             STRING,
  VISIBILITY         STRING,
  WEIGHT             DECIMAL(18,4),
  ID                 BIGINT,
  PARENT_SKU         STRING,
  PARENT_RX_EBS_PRODUCT_CODE STRING,
  CREATED_AT         TIMESTAMP,
  UPDATED_AT         TIMESTAMP,
  STOCK_QTY          DECIMAL(18,4),
  RX_LINK_TYPE       STRING
) USING DELTA;

## Step 4: Populate C$ Staging Table
Insert deduplicated records from `WC_MERCURY_PRODUCT_TS` into the staging table.
ROW_NUMBER partitioned by ID, ordered by INT_INSERT_DATE DESC ensures only the latest version per product is retained (RNK = 1).

In [ ]:
%sql
-- Step 4a: Insert deduplicated source data into staging
INSERT INTO workspace.PRXBI_DW.c_0filter_stg
SELECT
  NAME,
  PRICE,
  STATUS,
  VISIBILITY,
  WEIGHT,
  ID,
  PARENT_SKU,
  PARENT_RX_EBS_PRODUCT_CODE,
  CREATED_AT,
  UPDATED_AT,
  STOCK_QTY,
  RX_LINK_TYPE
FROM (
  SELECT
    NAME,
    PARENT_PRICE AS PRICE,
    STATUS,
    VISIBILITY,
    WEIGHT,
    ID,
    PARENT_SKU,
    PARENT_RX_EBS_PRODUCT_CODE,
    CREATED_AT,
    UPDATED_AT,
    STOCK_QTY,
    RX_LINK_TYPE,
    ROW_NUMBER() OVER (PARTITION BY ID ORDER BY INT_INSERT_DATE DESC) AS RNK
  FROM workspace.PRXBI_TS.WC_MERCURY_PRODUCT_TS
) dedup
WHERE RNK = 1;

In [ ]:
%sql
-- Step 4b: Validate staging record count
SELECT COUNT(*) AS staging_record_count FROM workspace.PRXBI_DW.c_0filter_stg;

## Step 5: Create I$ Flow Table
Drop and recreate the flow table `i_wc_badge_product_d_flow` which holds the change-detected records
with an IND_UPDATE flag ('I' for insert, 'U' for update).

In [ ]:
%sql
-- Step 5a: Drop flow table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_badge_product_d_flow;

In [ ]:
%sql
-- Step 5b: Create flow table
CREATE TABLE workspace.PRXBI_DW.i_wc_badge_product_d_flow (
  ROW_WID                    BIGINT,
  ID                         BIGINT,
  SKU                        STRING,
  NAME                       STRING,
  PRICE                      DECIMAL(18,4),
  STATUS                     STRING,
  VISIBILITY                 STRING,
  WEIGHT                     DECIMAL(18,4),
  STOCK_QTY                  DECIMAL(18,4),
  INTEGRATION_ID             BIGINT,
  DATASOURCE_NUM_ID          INT,
  W_INSERT_DT                TIMESTAMP,
  W_UPDATE_DT                TIMESTAMP,
  ETL_PROC_WID               BIGINT,
  PARENT_RX_EBS_PRODUCT_CODE STRING,
  CHANGED_ON_DT              TIMESTAMP,
  CREATED_ON_DT              TIMESTAMP,
  RX_LINK_TYPE               STRING,
  IND_UPDATE                 STRING
) USING DELTA;

## Step 6: Populate Flow Table with NOT EXISTS Change Detection
Insert records from the staging table into the flow table where no matching record (with identical column values)
exists in the target. Uses Spark NULL-safe equality operator `<=>` for all column comparisons.

In [ ]:
%sql
-- Step 6a: Insert into flow table with NOT EXISTS change detection
INSERT INTO workspace.PRXBI_DW.i_wc_badge_product_d_flow
SELECT
  NULL AS ROW_WID,
  S.ID,
  S.PARENT_SKU AS SKU,
  S.NAME,
  S.PRICE,
  S.STATUS,
  S.VISIBILITY,
  S.WEIGHT,
  S.STOCK_QTY,
  S.ID AS INTEGRATION_ID,
  380 AS DATASOURCE_NUM_ID,
  CURRENT_TIMESTAMP() AS W_INSERT_DT,
  CURRENT_TIMESTAMP() AS W_UPDATE_DT,
  CAST(${ETL_PROC_WID} AS BIGINT) AS ETL_PROC_WID,
  S.PARENT_RX_EBS_PRODUCT_CODE,
  S.UPDATED_AT AS CHANGED_ON_DT,
  S.CREATED_AT AS CREATED_ON_DT,
  S.RX_LINK_TYPE,
  'I' AS IND_UPDATE
FROM workspace.PRXBI_DW.c_0filter_stg S
WHERE NOT EXISTS (
  SELECT 1
  FROM workspace.PRXBI_DW.wc_badge_product_d T
  WHERE T.INTEGRATION_ID <=> S.ID
    AND T.DATASOURCE_NUM_ID <=> 380
    AND T.SKU <=> S.PARENT_SKU
    AND T.NAME <=> S.NAME
    AND T.PRICE <=> S.PRICE
    AND T.STATUS <=> S.STATUS
    AND T.VISIBILITY <=> S.VISIBILITY
    AND T.WEIGHT <=> S.WEIGHT
    AND T.STOCK_QTY <=> S.STOCK_QTY
    AND T.W_UPDATE_DT <=> S.UPDATED_AT
    AND T.PARENT_RX_EBS_PRODUCT_CODE <=> S.PARENT_RX_EBS_PRODUCT_CODE
    AND T.CHANGED_ON_DT <=> S.UPDATED_AT
    AND T.CREATED_ON_DT <=> S.CREATED_AT
    AND T.RX_LINK_TYPE <=> S.RX_LINK_TYPE
);

In [ ]:
%sql
-- Step 6b: Validate flow table record count
SELECT COUNT(*) AS flow_record_count FROM workspace.PRXBI_DW.i_wc_badge_product_d_flow;

## Step 7: Flag Updates in Flow Table
Set `IND_UPDATE = 'U'` for records in the flow table where a matching (INTEGRATION_ID, DATASOURCE_NUM_ID)
already exists in the target table. These records will be applied as updates rather than inserts.

In [ ]:
%sql
-- Step 7a: Flag records for update where they already exist in target
UPDATE workspace.PRXBI_DW.i_wc_badge_product_d_flow F
SET IND_UPDATE = 'U'
WHERE EXISTS (
  SELECT 1
  FROM workspace.PRXBI_DW.wc_badge_product_d T
  WHERE T.INTEGRATION_ID = F.INTEGRATION_ID
    AND T.DATASOURCE_NUM_ID = F.DATASOURCE_NUM_ID
);

In [ ]:
%sql
-- Step 7b: Validate IND_UPDATE breakdown
SELECT IND_UPDATE, COUNT(*) AS record_count
FROM workspace.PRXBI_DW.i_wc_badge_product_d_flow
GROUP BY IND_UPDATE
ORDER BY IND_UPDATE;

## Step 8: Apply Changes via MERGE INTO Target
MERGE replaces the separate UPDATE + INSERT from ODI.
- WHEN MATCHED AND IND_UPDATE = 'U': Update all columns in the target.
- WHEN NOT MATCHED AND IND_UPDATE = 'I': Insert new records (ROW_WID generated by IDENTITY column).

In [ ]:
%sql
-- Step 8: MERGE INTO target from flow table
MERGE INTO workspace.PRXBI_DW.wc_badge_product_d T
USING workspace.PRXBI_DW.i_wc_badge_product_d_flow S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
  AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID

WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
  T.ID                         = S.ID,
  T.SKU                        = S.SKU,
  T.NAME                       = S.NAME,
  T.PRICE                      = S.PRICE,
  T.STATUS                     = S.STATUS,
  T.VISIBILITY                 = S.VISIBILITY,
  T.WEIGHT                     = S.WEIGHT,
  T.STOCK_QTY                  = S.STOCK_QTY,
  T.W_UPDATE_DT                = S.W_UPDATE_DT,
  T.ETL_PROC_WID               = CAST(${ETL_PROC_WID} AS BIGINT),
  T.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE,
  T.CHANGED_ON_DT              = S.CHANGED_ON_DT,
  T.CREATED_ON_DT              = S.CREATED_ON_DT,
  T.RX_LINK_TYPE               = S.RX_LINK_TYPE

WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
  ID,
  SKU,
  NAME,
  PRICE,
  STATUS,
  VISIBILITY,
  WEIGHT,
  STOCK_QTY,
  INTEGRATION_ID,
  DATASOURCE_NUM_ID,
  W_INSERT_DT,
  W_UPDATE_DT,
  ETL_PROC_WID,
  PARENT_RX_EBS_PRODUCT_CODE,
  CHANGED_ON_DT,
  CREATED_ON_DT,
  RX_LINK_TYPE
) VALUES (
  S.ID,
  S.SKU,
  S.NAME,
  S.PRICE,
  S.STATUS,
  S.VISIBILITY,
  S.WEIGHT,
  S.STOCK_QTY,
  S.INTEGRATION_ID,
  S.DATASOURCE_NUM_ID,
  S.W_INSERT_DT,
  S.W_UPDATE_DT,
  CAST(${ETL_PROC_WID} AS BIGINT),
  S.PARENT_RX_EBS_PRODUCT_CODE,
  S.CHANGED_ON_DT,
  S.CREATED_ON_DT,
  S.RX_LINK_TYPE
);

## Step 9: Optimize Target Table
Replace Oracle `DBMS_STATS.GATHER_TABLE_STATS` with Delta `OPTIMIZE` and `ZORDER BY` for query performance.

In [ ]:
%sql
-- Step 9a: Optimize target table (compacts small files)
OPTIMIZE workspace.PRXBI_DW.wc_badge_product_d;

In [ ]:
%sql
-- Step 9b: Optimize with ZORDER for frequently filtered columns
OPTIMIZE workspace.PRXBI_DW.wc_badge_product_d
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## Step 10: Cleanup Temporary Tables
Drop the staging (C$) and flow (I$) tables used during the ETL process.

In [ ]:
%sql
-- Step 10a: Drop flow table
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_badge_product_d_flow;

In [ ]:
%sql
-- Step 10b: Drop staging table
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0filter_stg;

## Step 11: Final Validation
Verify the target table record count and inspect sample records to confirm the load completed successfully.

In [ ]:
%sql
-- Step 11a: Final validation - record count
SELECT
  COUNT(*) AS total_records,
  COUNT(DISTINCT INTEGRATION_ID) AS distinct_products,
  MIN(W_INSERT_DT) AS earliest_insert,
  MAX(W_UPDATE_DT) AS latest_update
FROM workspace.PRXBI_DW.wc_badge_product_d
WHERE DATASOURCE_NUM_ID = 380;

In [ ]:
%sql
-- Step 11b: Sample records from target
SELECT
  ROW_WID,
  ID,
  SKU,
  NAME,
  PRICE,
  STATUS,
  VISIBILITY,
  WEIGHT,
  STOCK_QTY,
  INTEGRATION_ID,
  DATASOURCE_NUM_ID,
  W_INSERT_DT,
  W_UPDATE_DT,
  ETL_PROC_WID,
  PARENT_RX_EBS_PRODUCT_CODE,
  CHANGED_ON_DT,
  CREATED_ON_DT,
  RX_LINK_TYPE
FROM workspace.PRXBI_DW.wc_badge_product_d
WHERE DATASOURCE_NUM_ID = 380
ORDER BY W_UPDATE_DT DESC
LIMIT 20;

## Conversion Notes (ODI to Databricks)

| ODI / Oracle Construct | Databricks / Spark SQL Equivalent |
|---|---|
| `PRXBI_DW_SEP` schema | `workspace.PRXBI_DW` |
| `PRXBI_TS_SEP` schema | `workspace.PRXBI_TS` |
| `SYSTIMESTAMP` | `CURRENT_TIMESTAMP()` |
| `NVL(col, val)` | `COALESCE(col, val)` |
| `/*+ append */` hint | Removed (Delta handles append natively) |
| `NOLOGGING` | Removed (Delta manages transaction logs) |
| Oracle indexes | Removed (Delta uses Z-ORDER for data skipping) |
| `DBMS_STATS.GATHER_TABLE_STATS` | `OPTIMIZE` + `ZORDER BY` |
| `SEQ.NEXTVAL` for ROW_WID | `BIGINT GENERATED ALWAYS AS IDENTITY` on target table |
| `#GLOBAL.v_ETL_JOB_TYPE` | Widget `${ETL_JOB_TYPE}` |
| `#SALES_AND_MARKETING.ETLProcWID` | Widget `${ETL_PROC_WID}` |
| Separate UPDATE + INSERT | `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` |
| Oracle bind-variable INSERT | Direct `INSERT INTO ... SELECT` |
| `VARCHAR2` | `STRING` |
| `NUMBER(x,y)` | `DECIMAL(x,y)` or `BIGINT`/`INT` |
| `TIMESTAMP(7)` | `TIMESTAMP` |
| `(T.COL = S.COL) OR (T.COL IS NULL AND S.COL IS NULL)` | `T.COL <=> S.COL` (NULL-safe equality) |
| C$ temporary staging table | `workspace.PRXBI_DW.c_0filter_stg` (Delta) |
| I$ flow/integration table | `workspace.PRXBI_DW.i_wc_badge_product_d_flow` (Delta) |